# ObserveServer

FastAPI server that wraps a `NetObserver` and exposes REST + WebSocket endpoints.
Runs as a background asyncio task alongside the Net.

In [ ]:
#|default_exp observe.server

In [ ]:
#|export
import asyncio
import json
import logging

from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.middleware.cors import CORSMiddleware
import uvicorn
import httpx

from netrun.net._net._net import Net

from netrun_utils.observe.observer import NetObserver
from netrun_utils.observe.models import SendControlRequest, InjectDataRequest

logger = logging.getLogger(__name__)

In [ ]:
#|export
class ObserveServer:
    """Non-blocking FastAPI server for observing and controlling a running Net.

    Exposes REST endpoints and a WebSocket endpoint for live state updates.
    Runs as a background asyncio task so the Net can continue executing.

    Usage::

        server = ObserveServer(net, port=8000)
        await server.start()
        # ... net runs ...
        await server.stop()

    Or as a context manager::

        async with ObserveServer(net, port=8000) as server:
            # ... net runs ...
    """

    def __init__(
        self,
        net: Net,
        host: str = "127.0.0.1",
        port: int = 8000,
        ws_interval: float = 0.2,
        name: str = "unnamed-net",
        registry_url: str | None = "http://localhost:18400",
    ):
        self._observer = NetObserver(net)
        self._host = host
        self._port = port
        self._ws_interval = ws_interval
        self._name = name
        self._registry_url = registry_url
        self._app = self._create_app()
        self._server: uvicorn.Server | None = None
        self._task: asyncio.Task | None = None
        self._heartbeat_task: asyncio.Task | None = None

    def _create_app(self) -> FastAPI:
        app = FastAPI(title="netrun-utils observe", version="0.1.0")

        app.add_middleware(
            CORSMiddleware,
            allow_origins=["*"],
            allow_methods=["*"],
            allow_headers=["*"],
        )

        obs = self._observer

        @app.get("/health")
        def health():
            return {"status": "healthy"}

        @app.get("/config")
        def config():
            return obs.get_config()

        @app.get("/status")
        def status():
            return obs.get_status().model_dump()

        @app.get("/nodes")
        def nodes():
            return [n.model_dump() for n in obs.get_nodes()]

        @app.get("/nodes/{name}")
        def node(name: str):
            return obs.get_node(name).model_dump()

        @app.get("/edges")
        def edges():
            return [e.model_dump() for e in obs.get_edges()]

        @app.get("/epochs")
        def epochs():
            return [e.model_dump() for e in obs.get_epoch_logs()]

        @app.get("/logs")
        def logs():
            return [e.model_dump() for e in obs.get_all_logs()]

        @app.get("/nodes/{name}/logs")
        def node_logs(name: str):
            return [e.model_dump() for e in obs.get_node_logs(name)]

        @app.post("/nodes/{name}/enable")
        def enable_node(name: str):
            return obs.enable_node(name).model_dump()

        @app.post("/nodes/{name}/disable")
        def disable_node(name: str):
            return obs.disable_node(name).model_dump()

        @app.post("/control")
        def send_control(req: SendControlRequest):
            return obs.send_control(req.node_name, req.control_type, req.value).model_dump()

        @app.post("/inject")
        def inject_data(req: InjectDataRequest):
            return obs.inject_data(req.node_name, req.port_name, req.values).model_dump()

        @app.websocket("/ws")
        async def websocket_endpoint(ws: WebSocket):
            await ws.accept()
            try:
                while True:
                    state = {
                        "status": obs.get_status().model_dump(),
                        "nodes": [n.model_dump() for n in obs.get_nodes()],
                        "edges": [e.model_dump() for e in obs.get_edges()],
                        "epochs": [e.model_dump() for e in obs.get_epoch_logs()],
                        "logs": [l.model_dump() for l in obs.get_all_logs()],
                    }
                    await ws.send_text(json.dumps(state))
                    await asyncio.sleep(self._ws_interval)
            except WebSocketDisconnect:
                pass

        return app

    async def start(self) -> None:
        """Start the server as a background asyncio task.

        Waits until the server is ready to accept connections before returning.
        Registers with the dashboard registry if registry_url is set.
        """
        config = uvicorn.Config(
            self._app,
            host=self._host,
            port=self._port,
            log_level="warning",
        )
        self._server = uvicorn.Server(config)
        self._task = asyncio.create_task(self._server.serve())

        # Wait until the server is accepting connections
        while not self._server.started:
            await asyncio.sleep(0.05)

        # Register with dashboard and start heartbeat
        if self._registry_url:
            await self._register()
            self._heartbeat_task = asyncio.create_task(self._heartbeat_loop())

    async def stop(self) -> None:
        """Gracefully stop the server."""
        # Stop heartbeat
        if self._heartbeat_task is not None:
            self._heartbeat_task.cancel()
            try:
                await self._heartbeat_task
            except asyncio.CancelledError:
                pass
            self._heartbeat_task = None

        # Deregister from dashboard
        if self._registry_url:
            await self._deregister()

        if self._server is not None:
            self._server.should_exit = True
        if self._task is not None:
            await self._task
            self._task = None
        self._server = None

    @property
    def url(self) -> str:
        """Base URL of the running server."""
        return f"http://{self._host}:{self._port}"

    async def _register(self) -> None:
        """Register with the dashboard registry. Silent on failure."""
        try:
            async with httpx.AsyncClient(timeout=5.0) as client:
                await client.post(
                    f"{self._registry_url}/api/register",
                    json={"name": self._name, "url": self.url},
                )
        except Exception:
            logger.debug("Failed to register with dashboard at %s", self._registry_url)

    async def _deregister(self) -> None:
        """Deregister from the dashboard registry. Silent on failure."""
        try:
            async with httpx.AsyncClient(timeout=5.0) as client:
                await client.post(
                    f"{self._registry_url}/api/deregister",
                    json={"url": self.url},
                )
        except Exception:
            logger.debug("Failed to deregister from dashboard at %s", self._registry_url)

    async def _heartbeat_loop(self) -> None:
        """Send heartbeats to the dashboard registry every 10 seconds."""
        while True:
            await asyncio.sleep(10)
            try:
                async with httpx.AsyncClient(timeout=5.0) as client:
                    await client.post(
                        f"{self._registry_url}/api/heartbeat",
                        json={"url": self.url},
                    )
            except Exception:
                logger.debug("Heartbeat to dashboard failed")

    async def __aenter__(self) -> "ObserveServer":
        await self.start()
        return self

    async def __aexit__(self, *exc) -> None:
        await self.stop()